# 🐐 GOAT-Net — Phase 1: Modern Data Collection
**Notebook:** `0_data_collection.ipynb`

This notebook:
1. Mounts Google Drive and creates the folder structure
2. Collects StatsBomb Open Data (lineups + filtered events)
3. Collects FBref & Understat season stats via `soccerdata`
4. Saves everything to Drive (heavy) and repo (light CSVs)
5. Generates a dataset inventory

**Architecture:** GitHub (code) → Colab (compute) → Google Drive (data). Large files never go to GitHub — all heavy data lives on Drive.

**v2 changes from the first draft:** every download step now checks whether the file already exists on Drive before fetching, so reruns are cheap and interrupted sessions don't waste API calls. Cell 5 is deliberately limited to the first 8 competitions as a speed test — remove `.head(8)` once you've confirmed it runs cleanly. Cell 10 no longer auto-pushes to GitHub; it prints the commands so you can review and run them yourself.
> **Design rule:** Large files never go to GitHub. Everything heavy lives on Google Drive.

### Cell 1: Setup & Install

In [15]:
# ============================================================
# GOAT-Net — Phase 0: Data Collection (Fixed Setup)
# ============================================================
import os, sys, random
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 1. Mount Drive & Load Userdata (for Secrets)
from google.colab import drive, userdata
drive.mount("/content/drive", force_remount=False)

# 2. Set Working Directory (NO GIT PULLING)
PROJECT = "GOAT-Net"
REPO = Path("/content/drive/MyDrive") / PROJECT
os.chdir(REPO)
print(f"📁 Working directory securely set to: {REPO}")

# 3. Fix Python Path & Clear Cache
SRC_DIR = REPO / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

for m in list(sys.modules):
    if m == "data" or m.startswith("data.") or m.startswith("goatnet."):
        del sys.modules[m]

# 4. Install Scraping Dependencies
print("⏳ Installing scraping dependencies...")
!pip install -q statsbombpy pyarrow beautifulsoup4 requests curl_cffi tqdm

# 5. Import Scraping Libraries
import io, re, json, time, shutil, datetime, csv, urllib.parse
import requests
from bs4 import BeautifulSoup, Comment
from tqdm.notebook import tqdm

# 6. Folder Architecture
ROOT = REPO
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
CACHE = ROOT / "cache"
META = ROOT / "metadata"

VAULT_FOLDERS = [
    DATA_RAW / "modern" / "statsbomb",
    DATA_RAW / "modern" / "fbref",
    DATA_RAW / "modern" / "understat",
    DATA_RAW / "modern" / "soccernet",
    DATA_RAW / "modern" / "transfermarkt",
    DATA_RAW / "legacy" / "goat_canon",
    DATA_RAW / "legacy" / "rsssf",
    DATA_RAW / "legacy" / "iffhs",
    DATA_RAW / "legacy" / "wikipedia",
    DATA_RAW / "legacy" / "worldfootball",
    DATA_RAW / "optional" / "fifa",
    DATA_RAW / "optional" / "fivethirtyeight",
    DATA_PROCESSED / "statsbomb",
    DATA_PROCESSED / "fbref",
    DATA_PROCESSED / "understat",
    CACHE,
    META,
]
for folder in VAULT_FOLDERS:
    folder.mkdir(parents=True, exist_ok=True)

# 7. Secrets Management (Strict Colab Secrets Only)
try:
    SCRAPER_API_KEY = userdata.get("SCRAPER_API_KEY")
    if not SCRAPER_API_KEY:
        raise ValueError("empty secret")
except Exception:
    raise RuntimeError(
        "SCRAPER_API_KEY not found in Colab Secrets. Add it via the key icon in the "
        "left sidebar, enable notebook access, then rerun this cell."
    )

try:
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    if not os.environ["KAGGLE_USERNAME"] or not os.environ["KAGGLE_KEY"]:
        raise ValueError("empty secret")
except Exception:
    raise RuntimeError(
        "KAGGLE_USERNAME and/or KAGGLE_KEY not found in Colab Secrets. Open your "
        "kaggle.json (downloaded from Kaggle Account settings), and add its 'username' "
        "and 'key' fields as two separate Colab Secrets named KAGGLE_USERNAME and KAGGLE_KEY."
    )

print(f"✅ Vault created and verified at: {ROOT}")
print("✅ Secrets loaded: SCRAPER_API_KEY, KAGGLE_USERNAME, KAGGLE_KEY")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📁 Working directory securely set to: /content/drive/MyDrive/GOAT-Net
⏳ Installing scraping dependencies...
✅ Vault created and verified at: /content/drive/MyDrive/GOAT-Net
✅ Secrets loaded: SCRAPER_API_KEY, KAGGLE_USERNAME, KAGGLE_KEY


### Cell 2: Mount Drive & Create Folders

In [16]:
# from google.colab import drive, userdata
# drive.mount("/content/drive")

# import io, re, json, time, shutil, datetime, csv, urllib.parse
# import requests
# import pandas as pd
# import numpy as np
# from pathlib import Path
# from bs4 import BeautifulSoup, Comment
# from tqdm.notebook import tqdm
# import warnings
# warnings.filterwarnings("ignore")

# # ===== CONFIGURATION =====
# ROOT = Path("/content/drive/MyDrive/GOAT-Net")
# DATA_RAW = ROOT / "data" / "raw"
# DATA_PROCESSED = ROOT / "data" / "processed"
# CACHE = ROOT / "cache"
# META = ROOT / "metadata"

# VAULT_FOLDERS = [
#     DATA_RAW / "modern" / "statsbomb",
#     DATA_RAW / "modern" / "fbref",
#     DATA_RAW / "modern" / "understat",
#     DATA_RAW / "modern" / "soccernet",
#     DATA_RAW / "modern" / "transfermarkt",
#     DATA_RAW / "legacy" / "goat_canon",
#     DATA_RAW / "legacy" / "rsssf",
#     DATA_RAW / "legacy" / "iffhs",
#     DATA_RAW / "legacy" / "wikipedia",
#     DATA_RAW / "legacy" / "worldfootball",
#     DATA_RAW / "optional" / "fifa",
#     DATA_RAW / "optional" / "fivethirtyeight",
#     DATA_PROCESSED / "statsbomb",
#     DATA_PROCESSED / "fbref",
#     DATA_PROCESSED / "understat",
#     CACHE,
#     META,
# ]
# for folder in VAULT_FOLDERS:
#     folder.mkdir(parents=True, exist_ok=True)

# # SECURITY: both secrets come ONLY from Colab Secrets, no fallbacks. A fallback value
# # sitting in this file is stolen the moment this notebook is pushed to a public repo.

# try:
#     SCRAPER_API_KEY = userdata.get("SCRAPER_API_KEY")
#     if not SCRAPER_API_KEY:
#         raise ValueError("empty secret")
# except Exception:
#     raise RuntimeError(
#         "SCRAPER_API_KEY not found in Colab Secrets. Add it via the key icon in the "
#         "left sidebar, enable notebook access, then rerun this cell."
#     )

# # Kaggle's CLI/library authenticates via KAGGLE_USERNAME + KAGGLE_KEY specifically --
# # NOT a single combined token. These come from the kaggle.json file Kaggle gives you
# # under Account -> Create New API Token (it has "username" and "key" fields).
# try:
#     os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
#     os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
#     if not os.environ["KAGGLE_USERNAME"] or not os.environ["KAGGLE_KEY"]:
#         raise ValueError("empty secret")
# except Exception:
#     raise RuntimeError(
#         "KAGGLE_USERNAME and/or KAGGLE_KEY not found in Colab Secrets. Open your "
#         "kaggle.json (downloaded from Kaggle Account settings), and add its 'username' "
#         "and 'key' fields as two separate Colab Secrets named KAGGLE_USERNAME and KAGGLE_KEY."
#     )

# print(f"Vault created and verified at: {ROOT}")
# print("Secrets loaded: SCRAPER_API_KEY, KAGGLE_USERNAME, KAGGLE_KEY")

### Cell 3: Tier‑1 Panel Definition

In [17]:
# Exactly as in docs/PREREGISTRATION.md section 3
TIER1_PANEL = [
    "Lionel Messi", "Cristiano Ronaldo", "Kylian Mbappé", "Neymar",
    "Kevin De Bruyne", "Robert Lewandowski", "Luka Modrić", "Erling Haaland",
]
# StatsBomb often stores full legal names rather than common names
STATSBOMB_PANEL = [
    "Lionel Andrés Messi Cuccittini", "Cristiano Ronaldo dos Santos Aveiro",
    "Kylian Mbappé Lottin", "Neymar da Silva Santos Junior",
    "Kevin De Bruyne", "Robert Lewandowski", "Luka Modrić", "Erling Haaland",
]
LEGACY_PANEL = [
    "Pelé", "Diego Maradona", "Johan Cruyff", "Franz Beckenbauer",
    "Alfredo Di Stéfano", "Ferenc Puskás",
]

print("Tier-1 Panel Active:", TIER1_PANEL)
print("Legacy Panel Active:", LEGACY_PANEL)

Tier-1 Panel Active: ['Lionel Messi', 'Cristiano Ronaldo', 'Kylian Mbappé', 'Neymar', 'Kevin De Bruyne', 'Robert Lewandowski', 'Luka Modrić', 'Erling Haaland']
Legacy Panel Active: ['Pelé', 'Diego Maradona', 'Johan Cruyff', 'Franz Beckenbauer', 'Alfredo Di Stéfano', 'Ferenc Puskás']


### Cell 4: StatsBomb – Competitions & Match Filtering

In [18]:
from statsbombpy import sb

competitions = sb.competitions()
CANDIDATE_COMPS = competitions[
    competitions["competition_name"].isin([
        "La Liga", "Premier League", "FIFA World Cup", "UEFA Euro",
        "Champions League", "Copa America", "Bundesliga", "Serie A"
    ])
]
print(f"Selected {len(CANDIDATE_COMPS)} candidate competition-seasons.")

Selected 51 candidate competition-seasons.


### Cell 5: StatsBomb – Collect Lineups (Find Panel Players)

In [19]:
LINEUPS_PATH = DATA_RAW / "modern" / "statsbomb" / "statsbomb_lineups_raw.csv"

if LINEUPS_PATH.exists():
    print("Lineups already exist on Drive. Loading cached copy...")
    lineups_df = pd.read_csv(LINEUPS_PATH)
else:
    print("Downloading StatsBomb lineups across competitions...")
    all_lineups = []
    for _, row in tqdm(CANDIDATE_COMPS.iterrows(), total=len(CANDIDATE_COMPS), desc="Scanning competitions"):
        comp_id, season_id = row["competition_id"], row["season_id"]
        try:
            matches = sb.matches(competition_id=comp_id, season_id=season_id)
        except Exception:
            continue
        for match_id in matches["match_id"]:
            try:
                lineup = sb.lineups(match_id=match_id)
                for team, df in lineup.items():
                    df = df.copy()
                    df["match_id"] = match_id
                    df["competition_name"] = row["competition_name"]
                    df["season_name"] = row["season_name"]
                    all_lineups.append(df)
            except Exception:
                continue
            time.sleep(0.05)
    lineups_df = pd.concat(all_lineups, ignore_index=True) if all_lineups else pd.DataFrame()
    if not lineups_df.empty:
        lineups_df.to_csv(LINEUPS_PATH, index=False)
        print(f"Saved {len(lineups_df)} lineup rows to Drive.")

print(f"Total matches indexed: {lineups_df['match_id'].nunique() if not lineups_df.empty else 0}")


Lineups already exist on Drive. Loading cached copy...
Total matches indexed: 1965


### Cell 6: Filter to Panel Players & Show Coverage

In [20]:
if not lineups_df.empty:
    panel_matches = lineups_df[
        lineups_df["player_name"].isin(TIER1_PANEL) | lineups_df["player_name"].isin(STATSBOMB_PANEL)
    ].copy()
    coverage = panel_matches.groupby("player_name")["match_id"].nunique().sort_values(ascending=False)
else:
    panel_matches = pd.DataFrame()
    coverage = pd.Series(dtype=int)

PANEL_MATCHES_PATH = DATA_RAW / "modern" / "statsbomb" / "statsbomb_panel_matches.csv"
panel_matches.to_csv(PANEL_MATCHES_PATH, index=False)

print("\nStatsBomb Coverage per player:")
display(coverage)


StatsBomb Coverage per player:


,match_id
player_name,
Lionel Andrés Messi Cuccittini,539
Neymar da Silva Santos Junior,124
Cristiano Ronaldo dos Santos Aveiro,76
Luka Modrić,72
Kevin De Bruyne,44
Kylian Mbappé Lottin,24
Robert Lewandowski,14


### Cell 7: StatsBomb – Download Event Data for Panel Matches

In [21]:
EVENTS_PATH = DATA_RAW / "modern" / "statsbomb" / "statsbomb_panel_events.parquet"

if EVENTS_PATH.exists():
    print("Events dataset already exists on Drive. Loading cached copy...")
    all_events = pd.read_parquet(EVENTS_PATH)
else:
    print("Downloading filtered spatial events (Shots & Passes)...")
    event_records = []
    panel_match_ids = panel_matches["match_id"].unique() if not panel_matches.empty else []
    for match_id in tqdm(panel_match_ids, desc="Downloading Events"):
        try:
            events = sb.events(match_id=match_id)
            mask = (
                (events["player"].isin(TIER1_PANEL) | events["player"].isin(STATSBOMB_PANEL)) &
                events["type"].isin(["Shot", "Pass"])
            )
            filtered = events[mask].copy()
            if not filtered.empty:
                filtered["match_id"] = match_id
                event_records.append(filtered)
        except Exception:
            continue
    if event_records:
        all_events = pd.concat(event_records, ignore_index=True)
        all_events.to_parquet(EVENTS_PATH, index=False)
        print(f"Saved {len(all_events)} filtered events to {EVENTS_PATH}")
    else:
        all_events = pd.DataFrame()
        print("No panel events found.")


Events dataset already exists on Drive. Loading cached copy...


### Cell 8: FBref & Understat Collection

In [22]:
# ============================================================
# GOAT-Net — Cell 8: FBref Full Career Season Stats (Smart Deferral Engine)
# ============================================================
import io
import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup

FBREF_DIR = DATA_RAW / "modern" / "fbref"
FBREF_DIR.mkdir(parents=True, exist_ok=True)

# Perfectly aligned with PREREGISTRATION.md (Tier 1 - Modern)
TIER1_FBREF_PROFILES = {
    "Lionel Messi": "d70ce98e/Lionel-Messi",
    "Cristiano Ronaldo": "dea698d9/Cristiano-Ronaldo",
    "Kylian Mbappé": "42fd9c7f/Kylian-Mbappe",
    "Neymar": "69235f98/Neymar",
    "Kevin De Bruyne": "e46012d4/Kevin-De-Bruyne",
    "Robert Lewandowski": "8d78e732/Robert-Lewandowski",
    "Luka Modrić": "6025fab1/Luka-Modric",
    "Erling Haaland": "1f44ac21/Erling-Haaland"
}

CAPTCHA_MARKERS = [
    "just a moment", "cf-challenge", "cloudflare",
    "enable javascript", "attention required", "captcha",
    "security check"
]

# Global flag to dynamically disable premium if ScraperAPI account doesn't support it
premium_supported = True

def is_captcha_or_block(html_text):
    """Detects if the returned HTML is a Cloudflare/CAPTCHA challenge page."""
    if not html_text or len(html_text) < 1000:
        return True
    text_lower = html_text.lower()
    return any(marker in text_lower for marker in CAPTCHA_MARKERS)

def parse_career_table(html_text):
    """Uncomments HTML, isolates the career table, and parses using bs4 engine."""
    uncommented_html = html_text.replace("<!--", "").replace("-->", "")
    soup = BeautifulSoup(uncommented_html, "html.parser")

    target_table = None
    for table in soup.find_all("table"):
        table_str = str(table)
        if "Season" in table_str and "Comp" in table_str and "Squad" in table_str:
            target_table = table
            break

    if target_table is None:
        return None

    tables = pd.read_html(io.StringIO(str(target_table)), flavor="bs4")
    if not tables:
        return None

    df = tables[0]
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = ["_".join(col).strip() if "Unnamed" not in col[0] else col[1] for col in df.columns.values]

    return df

def fetch_and_parse_player(player_name, profile_slug, max_retries=3):
    """Fetches URL with smart logging, self-healing params, and jittered backoff."""
    global premium_supported
    url_name = profile_slug.split('/')[1]
    target_url = f"https://fbref.com/en/players/{profile_slug}/all_comps/{url_name}-Stats---All-Competitions"

    for attempt in range(1, max_retries + 1):
        payload = {"api_key": SCRAPER_API_KEY, "url": target_url}

        # Only try premium if we haven't proven it breaks the account
        if attempt >= 2 and premium_supported:
            payload["premium"] = "true"

        try:
            response = requests.get("https://api.scraperapi.com/", params=payload, timeout=60)

            if response.status_code == 200:
                if is_captcha_or_block(response.text):
                    print(f"    ⚠️ Attempt {attempt}/{max_retries}: Disguised CAPTCHA detected (HTTP 200).")
                else:
                    return parse_career_table(response.text)

            elif response.status_code == 500 and "premium" in payload:
                print(f"    ⚠️ Attempt {attempt}/{max_retries}: HTTP 500 triggered by premium=true. Disabling premium globally.")
                premium_supported = False
            else:
                print(f"    ⚠️ Attempt {attempt}/{max_retries}: HTTP {response.status_code} error.")

        except Exception as err:
            print(f"    ⚠️ Attempt {attempt}/{max_retries} Network Error: {err}")

        # If we reach here, it failed. Apply jittered backoff before next attempt.
        if attempt < max_retries:
            backoff = (attempt * 4) + random.uniform(1.0, 3.5)
            print(f"       Retrying in {backoff:.1f}s...")
            time.sleep(backoff)

    return None # Completely failed after max_retries

def clean_and_save(df, player_name, out_path):
    """Cleans FBref formatting quirks and saves to Parquet."""
    df = df[df["Season"].notna()].copy()
    df = df[df["Season"] != "Season"].copy()
    df = df.dropna(subset=["Comp"]).copy()
    df = df[~df["Comp"].str.contains("Total|Career|Seasons", case=False, na=False)].copy()
    df["Player"] = player_name
    df.to_parquet(out_path, index=False)
    print(f"    ✅ Saved {len(df)} season/competition records -> {out_path.name}")

# --- MAIN EXECUTION ---
print("\n=== FBref Full Career Season Stats Extraction ===")
deferred_queue = {}

# Pass 1: Standard Extraction
for player_name, profile_slug in TIER1_FBREF_PROFILES.items():
    out_path = FBREF_DIR / f"fbref_career_stats_{player_name.replace(' ', '_')}.parquet"

    if out_path.exists():
        print(f"  ⚡ Already exists on Drive (Skipping): {out_path.name}")
        continue

    print(f"  Scraping full career for: {player_name}...")
    df_stats = fetch_and_parse_player(player_name, profile_slug)

    if df_stats is not None and not df_stats.empty:
        clean_and_save(df_stats, player_name, out_path)
    else:
        print(f"    ⏳ Deferring {player_name} to the cooldown queue...")
        deferred_queue[player_name] = profile_slug

    time.sleep(random.uniform(4.0, 7.0)) # Polite spacing between different players

# Pass 2: Deferred Cooldown Queue
if deferred_queue:
    cooldown = 75
    print(f"\n⏳ Entering {cooldown}s cooldown to shed Cloudflare heat before processing {len(deferred_queue)} deferred players...")
    time.sleep(cooldown)

    print("\n=== Processing Deferred Players ===")
    for player_name, profile_slug in deferred_queue.items():
        out_path = FBREF_DIR / f"fbref_career_stats_{player_name.replace(' ', '_')}.parquet"
        print(f"  Final attempt for: {player_name}...")

        # Give them one last focused try with fewer retries
        df_stats = fetch_and_parse_player(player_name, profile_slug, max_retries=2)

        if df_stats is not None and not df_stats.empty:
            clean_and_save(df_stats, player_name, out_path)
        else:
            print(f"    ❌ Final failure for {player_name}. URL is heavily flagged by FBref.")

print("\n🎉 Full Career extraction run complete!")


=== FBref Full Career Season Stats Extraction ===
  ⚡ Already exists on Drive (Skipping): fbref_career_stats_Lionel_Messi.parquet
  ⚡ Already exists on Drive (Skipping): fbref_career_stats_Cristiano_Ronaldo.parquet
  ⚡ Already exists on Drive (Skipping): fbref_career_stats_Kylian_Mbappé.parquet
  ⚡ Already exists on Drive (Skipping): fbref_career_stats_Neymar.parquet
  Scraping full career for: Kevin De Bruyne...


KeyboardInterrupt: 

In [26]:
# ============================================================
# GOAT-Net — Local HTML Parser for FBref (The Cloudflare Bypass)
# ============================================================
import io
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

FBREF_DIR = DATA_RAW / "modern" / "fbref"

# Map the HTML filenames you uploaded to the player's official name
LOCAL_HTML_FILES = {
    "ronaldo.html": "Cristiano Ronaldo",
    "mbappe.html": "Kylian Mbappé",
    "neymar.html": "Neymar",
    "debruyne.html": "Kevin De Bruyne",
    "lewandowski.html": "Robert Lewandowski",
    "modric.html": "Luka Modrić",
    "haaland.html": "Erling Haaland"
}

def parse_local_html(filepath, player_name):
    with open(filepath, "r", encoding="utf-8") as f:
        html_text = f.read()

    uncommented_html = html_text.replace("<!--", "").replace("-->", "")

    # Read ALL tables from the HTML
    try:
        tables = pd.read_html(io.StringIO(uncommented_html), flavor="bs4")
    except ValueError:
        raise ValueError("No tables could be parsed from the HTML.")

    df = None
    for tbl in tables:
        # Flatten multi-level columns
        if isinstance(tbl.columns, pd.MultiIndex):
            tbl.columns = ["_".join(col).strip() if "Unnamed" not in col[0] else col[1] for col in tbl.columns.values]

        # Verify the actual DataFrame has our target columns
        if "Season" in tbl.columns and "Comp" in tbl.columns and "Squad" in tbl.columns:
            df = tbl
            break

    if df is None:
        raise ValueError(f"Could not find the career table with 'Season' column in {filepath.name}")

    df = df[df["Season"].notna()].copy()
    df = df[df["Season"] != "Season"].copy()
    df = df.dropna(subset=["Comp"]).copy()
    df = df[~df["Comp"].str.contains("Total|Career|Seasons", case=False, na=False)].copy()
    df["Player"] = player_name

    return df

print("=== Local FBref HTML Parser ===")
for filename, player_name in LOCAL_HTML_FILES.items():
    html_path = FBREF_DIR / filename
    out_path = FBREF_DIR / f"fbref_career_stats_{player_name.replace(' ', '_')}.parquet"

    if out_path.exists():
        print(f"  ⚡ Already exists: {out_path.name}")
        continue

    if html_path.exists():
        print(f"  Parsing local file: {filename}...")
        try:
            df = parse_local_html(html_path, player_name)
            df.to_parquet(out_path, index=False)
            print(f"    ✅ Saved {len(df)} records -> {out_path.name}")
        except Exception as e:
            print(f"    ❌ Failed to parse {filename}: {e}")
    else:
        print(f"  ⚠️ Missing {filename} in {FBREF_DIR}")

print("\n🎉 Local parsing complete!")

=== Local FBref HTML Parser ===
  ⚡ Already exists: fbref_career_stats_Cristiano_Ronaldo.parquet
  ⚡ Already exists: fbref_career_stats_Kylian_Mbappé.parquet
  ⚡ Already exists: fbref_career_stats_Neymar.parquet
  ⚡ Already exists: fbref_career_stats_Kevin_De_Bruyne.parquet
  Parsing local file: lewandowski.html...
    ✅ Saved 69 records -> fbref_career_stats_Robert_Lewandowski.parquet
  ⚡ Already exists: fbref_career_stats_Luka_Modrić.parquet
  ⚡ Already exists: fbref_career_stats_Erling_Haaland.parquet

🎉 Local parsing complete!


In [ ]:
# # ============================================================
# # FIXED UNDERSTAT SCRAPER (Phase 1C) - Official Kaggle API
# # ============================================================
# import os
# import pandas as pd
# from pathlib import Path
# from google.colab import userdata

# print("\n=== Phase 1C: Understat xG Extraction via Kaggle API ===")

# UNDERSTAT_SAVE_PATH = DATA_RAW / "modern" / "understat" / "tier1_understat_xg.parquet"

# if UNDERSTAT_SAVE_PATH.exists():
#     print("⚡ Understat dataset already exists on Drive. Skipping.")
# else:
#     # 1. Securely load the Kaggle API token from Colab Secrets
#     try:
#         os.environ["KAGGLE_API_TOKEN"] = userdata.get('KAGGLE_API_TOKEN')
#     except userdata.SecretNotFoundError:
#         raise ValueError("KAGGLE_API_TOKEN not found in Colab Secrets. Please add it via the 🔑 menu.")

#     print("Downloading the Understat Database using official Kaggle API...")

#     # 2. Download and unzip the dataset quietly using the Kaggle CLI
#     !kaggle datasets download mexwell/understat-database --unzip -p /content/kaggle_data/understat-database -q

#     # 3. Process the downloaded CSV files
#     kaggle_dir = Path("/content/kaggle_data/understat-database")
#     understat_records = []

#     # Find all CSVs related to players in the downloaded folder
#     player_csvs = list(kaggle_dir.rglob("*player*.csv")) + list(kaggle_dir.rglob("*shots*.csv"))

#     if not player_csvs:
#          print("❌ Could not find the CSV files. Kaggle download may have failed.")
#     else:
#         print(f"Processing {len(player_csvs)} CSV files for Panel Players...")
#         for csv_path in player_csvs:
#             try:
#                 # Load the CSV
#                 df_u = pd.read_csv(csv_path)

#                 # Different Kaggle authors use different column names. Find the player name column.
#                 name_col = None
#                 for col in ["player_name", "Player", "player"]:
#                     if col in df_u.columns:
#                         name_col = col
#                         break

#                 if name_col:
#                     # Filter for our TIER1_PANEL players
#                     mask = df_u[name_col].astype(str).str.contains('|'.join(TIER1_PANEL), case=False, na=False)
#                     p_u = df_u[mask]

#                     if not p_u.empty:
#                         # Append the file's folder name (usually the league/season) as context
#                         p_u["Source_File"] = csv_path.name
#                         understat_records.append(p_u)
#             except Exception as e:
#                 continue

#         # 4. Combine and Save safely to Google Drive
#         if understat_records:
#             final_understat = pd.concat(understat_records, ignore_index=True)

#             # Remove any duplicate rows just in case
#             final_understat = final_understat.drop_duplicates()

#             final_understat.to_parquet(UNDERSTAT_SAVE_PATH, index=False)
#             print(f"\n🎉 SUCCESS! Extracted {len(final_understat)} Understat xG records to Drive.")
#             display(final_understat.head())
#         else:
#             print("\n⚠️ No panel players found in the Kaggle dataset.")

#     # Cleanup memory by deleting the massive raw CSVs from Colab
#     import shutil
#     shutil.rmtree('/content/kaggle_data', ignore_errors=True)

In [ ]:
# # ============================================================
# # GOAT-Net — Phase 2: Finishing Pending & Optional Datasets
# # ============================================================
# import os
# import time
# import shutil
# import requests
# import pandas as pd
# from pathlib import Path
# from google.colab import userdata

# # Install wikipedia library for NLP data
# !pip install -q wikipedia
# import wikipedia

# # Define paths
# ROOT = Path("/content/drive/MyDrive/GOAT-Net")
# DATA_RAW = ROOT / "data" / "raw"
# MODERN_DIR = DATA_RAW / "modern"
# LEGACY_DIR = DATA_RAW / "legacy"

# # Create a new Optional directory for Tier 3 data
# OPTIONAL_DIR = DATA_RAW / "optional"
# for folder in ["fifa", "fivethirtyeight"]:
#     (OPTIONAL_DIR / folder).mkdir(parents=True, exist_ok=True)

# TM_DIR = MODERN_DIR / "transfermarkt"
# TM_DIR.mkdir(parents=True, exist_ok=True)

# WIKI_DIR = LEGACY_DIR / "wikipedia"
# WIKI_DIR.mkdir(parents=True, exist_ok=True)

# # Panels
# TIER1_PANEL = [
#     "Lionel Messi", "Cristiano Ronaldo", "Kylian Mbappé", "Neymar",
#     "Kevin De Bruyne", "Robert Lewandowski", "Luka Modrić", "Erling Haaland"
# ]
# LEGACY_PANEL = [
#     "Pelé", "Diego Maradona", "Johan Cruyff", "Franz Beckenbauer",
#     "Alfredo Di Stéfano", "Ferenc Puskás"
# ]

# # Authenticate Kaggle automatically
# os.environ["KAGGLE_API_TOKEN"] = userdata.get('KAGGLE_API_TOKEN')

# # ==========================================
# # 1. TRANSFERMARKT (Tier 1 - Pending)
# # ==========================================
# print("\n=== 1. Transfermarkt Data Extraction ===")
# if not (TM_DIR / "tier1_tm_valuations.parquet").exists():
#     print("Downloading Transfermarkt Database from Kaggle...")
#     !kaggle datasets download davidcariboo/player-scores --unzip -p /content/kaggle_tm -q

#     try:
#         tm_players = pd.read_csv("/content/kaggle_tm/players.csv")
#         tm_vals = pd.read_csv("/content/kaggle_tm/player_valuations.csv")

#         # Match names for our Tier 1 Panel
#         mask = tm_players["name"].astype(str).str.contains('|'.join(TIER1_PANEL), case=False, na=False)
#         panel_tm = tm_players[mask]

#         if not panel_tm.empty:
#             panel_ids = panel_tm["player_id"].unique()
#             panel_vals = tm_vals[tm_vals["player_id"].isin(panel_ids)]

#             panel_tm.to_parquet(TM_DIR / "tier1_tm_players.parquet", index=False)
#             panel_vals.to_parquet(TM_DIR / "tier1_tm_valuations.parquet", index=False)
#             print(f"✅ Saved Transfermarkt data: {len(panel_tm)} profiles, {len(panel_vals)} valuation records.")
#     except Exception as e:
#         print(f"❌ Transfermarkt processing failed: {e}")

#     shutil.rmtree('/content/kaggle_tm', ignore_errors=True)
# else:
#     print("⚡ Transfermarkt data already exists on Drive.")

# # ==========================================
# # 2. EA SPORTS FIFA RATINGS (Tier 3 - Optional)
# # ==========================================
# print("\n=== 2. EA Sports FC / FIFA Ratings Extraction ===")
# FIFA_DIR = OPTIONAL_DIR / "fifa"
# if not (FIFA_DIR / "tier1_fifa_ratings.parquet").exists():
#     print("Downloading EA Sports Database from Kaggle...")
#     !kaggle datasets download stefanoleone992/ea-sports-fc-24-complete-player-dataset --unzip -p /content/kaggle_fifa -q

#     try:
#         fifa_df = pd.read_csv("/content/kaggle_fifa/male_players.csv", low_memory=False)
#         mask = fifa_df["short_name"].astype(str).str.contains('|'.join(TIER1_PANEL), case=False, na=False) | \
#                fifa_df["long_name"].astype(str).str.contains('|'.join(TIER1_PANEL), case=False, na=False)

#         panel_fifa = fifa_df[mask]
#         if not panel_fifa.empty:
#             panel_fifa.to_parquet(FIFA_DIR / "tier1_fifa_ratings.parquet", index=False)
#             print(f"✅ Saved FIFA data: {len(panel_fifa)} historical rating records.")
#     except Exception as e:
#         print(f"❌ FIFA processing failed: {e}")

#     shutil.rmtree('/content/kaggle_fifa', ignore_errors=True)
# else:
#     print("⚡ FIFA ratings already exist on Drive.")

# # ==========================================
# # 3. FIVETHIRTYEIGHT SPI (Tier 3 - Optional)
# # ==========================================
# print("\n=== 3. FiveThirtyEight SPI Team Rankings ===")
# FTE_DIR = OPTIONAL_DIR / "fivethirtyeight"
# if not (FTE_DIR / "spi_global_rankings.csv").exists():
#     print("Downloading 538 SPI from GitHub...")
#     try:
#         spi_url = "https://raw.githubusercontent.com/fivethirtyeight/data/master/soccer-spi/spi_global_rankings.csv"
#         spi_df = pd.read_csv(spi_url)
#         spi_df.to_csv(FTE_DIR / "spi_global_rankings.csv", index=False)
#         print(f"✅ Saved 538 SPI Rankings: {len(spi_df)} clubs.")
#     except Exception as e:
#         print(f"❌ 538 SPI failed: {e}")
# else:
#     print("⚡ 538 SPI already exists on Drive.")

# # ==========================================
# # 4. WIKIPEDIA BIOS (Tier 2 - Pending)
# # ==========================================
# print("\n=== 4. Wikipedia Biographies (NLP Engine) ===")
# if not (WIKI_DIR / "goat_biographies.parquet").exists():
#     ALL_PLAYERS = TIER1_PANEL + LEGACY_PANEL
#     wiki_data = []

#     for player in ALL_PLAYERS:
#         try:
#             print(f"  Fetching Wiki: {player}...")
#             # Download the textual summary of the player's career
#             summary = wikipedia.summary(player, auto_suggest=False)
#             wiki_data.append({"Player": player, "Biography_Summary": summary})
#             time.sleep(1)
#         except Exception as e:
#             print(f"    ⚠️ Could not fetch {player}: {e}")

#     if wiki_data:
#         wiki_df = pd.DataFrame(wiki_data)
#         wiki_df.to_parquet(WIKI_DIR / "goat_biographies.parquet", index=False)
#         print(f"✅ Saved {len(wiki_df)} biographies to Drive.")
# else:
#     print("⚡ Biographies already exist on Drive.")

# # ==========================================
# # 5. LEGACY CSV TEMPLATES (Tier 2 - Pending)
# # ==========================================
# print("\n=== 5. Generating Blank Legacy CSV Templates ===")
# import csv

# templates = {
#     "goat_canon/goat_canon_match_logs_template.csv": ["Player", "Date", "Opponent", "Competition", "Goals", "Assists", "Key_Plays", "Video_Available", "Notes"],
#     "rsssf/rsssf_career_goals_template.csv": ["Player", "Year", "Team", "Competition", "Official_Goals", "Friendly_Goals", "Notes"],
#     "iffhs/iffhs_awards_template.csv": ["Player", "Year", "Award_Name", "Rank", "Points"],
#     "worldfootball/worldfootball_match_history_template.csv": ["Player", "Date", "Team", "Opponent", "Result", "Goals", "Minutes_Played"]
# }

# for path_suffix, columns in templates.items():
#     full_path = LEGACY_DIR / path_suffix
#     full_path.parent.mkdir(parents=True, exist_ok=True)

#     if not full_path.exists():
#         with open(full_path, "w", newline="", encoding="utf-8") as f:
#             writer = csv.writer(f)
#             writer.writerow(columns)
#         print(f"✅ Created template: {path_suffix}")
#     else:
#         print(f"⚡ Template already exists: {path_suffix}")

# print("\n🎉 ALL PENDING & OPTIONAL DATA TASKS COMPLETED!")

In [ ]:
# # ============================================================
# # GOAT-Net — Patch: Fixing 538 SPI & Wikipedia
# # ============================================================
# import requests
# import urllib.parse
# import pandas as pd
# from pathlib import Path
# import time

# ROOT = Path("/content/drive/MyDrive/GOAT-Net")
# DATA_RAW = ROOT / "data" / "raw"
# FTE_DIR = DATA_RAW / "optional" / "fivethirtyeight"
# WIKI_DIR = DATA_RAW / "legacy" / "wikipedia"

# # 1. 538 Patch
# print("=== 1. PATCHING FIVETHIRTYEIGHT SPI ===")
# try:
#     # Pulling from the updated 538 API endpoint
#     spi_url = "https://projects.fivethirtyeight.com/soccer-api/club/spi_global_rankings.csv"
#     spi_df = pd.read_csv(spi_url)
#     spi_df.to_csv(FTE_DIR / "spi_global_rankings.csv", index=False)
#     print(f"✅ Saved 538 SPI Rankings: {len(spi_df)} clubs.")
# except Exception as e:
#     print(f"❌ 538 SPI failed: {e}")

# # 2. Wikipedia Patch
# print("\n=== 2. PATCHING WIKIPEDIA BIOGRAPHIES ===")
# wiki_data = []

# ALL_PLAYERS = [
#     "Lionel Messi", "Cristiano Ronaldo", "Kylian Mbappé", "Neymar",
#     "Kevin De Bruyne", "Robert Lewandowski", "Luka Modrić", "Erling Haaland",
#     "Pelé", "Diego Maradona", "Johan Cruyff", "Franz Beckenbauer",
#     "Alfredo Di Stéfano", "Ferenc Puskás"
# ]

# # Wikimedia strictly requires a User-Agent to prevent 403 Forbidden blocks
# headers = {
#     "User-Agent": "GOAT-Net/1.0 (atix.algo@gmail.com)"
# }

# for player in ALL_PLAYERS:
#     try:
#         print(f"  Fetching Wiki via REST API: {player}...")
#         formatted_name = urllib.parse.quote(player.replace(" ", "_"))
#         url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{formatted_name}"

#         res = requests.get(url, headers=headers, timeout=10)

#         if res.status_code == 200:
#             summary = res.json().get("extract")
#             wiki_data.append({"Player": player, "Biography_Summary": summary})
#         else:
#             # Fallback search if the exact name isn't the primary Wikipedia URL (e.g., naming variants)
#             search_url = f"https://en.wikipedia.org/w/api.php?action=query&list=search&srsearch={urllib.parse.quote(player)}&utf8=&format=json"
#             search_res = requests.get(search_url, headers=headers).json()

#             if search_res.get('query', {}).get('search'):
#                 best_match = search_res['query']['search'][0]['title']
#                 best_match_url = urllib.parse.quote(best_match.replace(" ", "_"))
#                 fallback_res = requests.get(f"https://en.wikipedia.org/api/rest_v1/page/summary/{best_match_url}", headers=headers)

#                 if fallback_res.status_code == 200:
#                     summary = fallback_res.json().get("extract")
#                     wiki_data.append({"Player": player, "Biography_Summary": summary})
#                     print(f"    └─ ✅ Fixed using search redirect -> {best_match}")
#             else:
#                 print(f"    ⚠️ No search results found.")

#         time.sleep(1)
#     except Exception as e:
#         print(f"    ❌ Could not fetch {player}: {e}")

# if wiki_data:
#     wiki_df = pd.DataFrame(wiki_data)
#     wiki_df.to_parquet(WIKI_DIR / "goat_biographies.parquet", index=False)
#     print(f"\n🎉 Saved {len(wiki_df)} accurate biographies to Drive.")

In [ ]:
# import os
# import shutil
# import pandas as pd
# from pathlib import Path
# from google.colab import userdata

# FTE_DIR = Path("/content/drive/MyDrive/GOAT-Net/data/raw/optional/fivethirtyeight")
# FTE_DIR.mkdir(parents=True, exist_ok=True)

# print("=== FINAL PATCH: FIVETHIRTYEIGHT SPI (KAGGLE ARCHIVE) ===")
# try:
#     # 1. Load the Kaggle API token from Colab Secrets (from Phase 1C)
#     os.environ["KAGGLE_API_TOKEN"] = userdata.get('KAGGLE_API_TOKEN')

#     print("Downloading the frozen 538 SPI Database from Kaggle...")
#     # 2. Download and unzip the archive silently
#     !kaggle datasets download thedevastator/club-soccer-predictions-spi-ratings-and-forecast --unzip -p /content/kaggle_spi -q

#     # 3. Move the data to your Google Drive Vault
#     spi_csv_path = "/content/kaggle_spi/spi_global_rankings.csv"
#     spi_df = pd.read_csv(spi_csv_path)
#     spi_df.to_csv(FTE_DIR / "spi_global_rankings.csv", index=False)

#     print(f"✅ Saved 538 SPI Rankings: {len(spi_df)} clubs.")

#     # 4. Clean up temporary files
#     shutil.rmtree('/content/kaggle_spi', ignore_errors=True)
# except Exception as e:
#     print(f"❌ 538 SPI failed: {e}")

### Cell 9: Generate Dataset Inventory

In [ ]:
# Static bulk mirror -- no live scraping, no IP-blocking risk, no ScraperAPI credits.
# Trade-off: may lag behind the current season; if a needed season is missing, that's a
# note for the dataset inventory, not a reason to silently fall back to scraping.

UNDERSTAT_SAVE_PATH = DATA_RAW / "modern" / "understat" / "tier1_understat_xg.parquet"

if UNDERSTAT_SAVE_PATH.exists():
    print("Understat dataset already exists on Drive. Skipping.")
else:
    print("Downloading Understat database from Kaggle...")
    !kaggle datasets download mexwell/understat-database --unzip -p /content/kaggle_data/understat-database -q

    kaggle_dir = Path("/content/kaggle_data/understat-database")
    understat_records = []
    player_csvs = list(kaggle_dir.rglob("*player*.csv")) + list(kaggle_dir.rglob("*shots*.csv"))

    if not player_csvs:
        print("Could not find expected CSV files -- Kaggle download may have failed. Check KAGGLE_USERNAME/KAGGLE_KEY.")
    else:
        print(f"Processing {len(player_csvs)} CSV files for panel players...")
        for csv_path in player_csvs:
            try:
                df_u = pd.read_csv(csv_path)
                name_col = next((c for c in ["player_name", "Player", "player"] if c in df_u.columns), None)
                if name_col:
                    mask = df_u[name_col].astype(str).str.contains("|".join(TIER1_PANEL), case=False, na=False)
                    p_u = df_u[mask]
                    if not p_u.empty:
                        p_u["Source_File"] = csv_path.name
                        understat_records.append(p_u)
            except Exception:
                continue

        if understat_records:
            final_understat = pd.concat(understat_records, ignore_index=True).drop_duplicates()
            final_understat.to_parquet(UNDERSTAT_SAVE_PATH, index=False)
            print(f"Saved {len(final_understat)} Understat records to Drive.")
            display(final_understat.head())
        else:
            print("No panel players found in the Kaggle dataset.")

    shutil.rmtree("/content/kaggle_data", ignore_errors=True)


### Cell 10: Prepare Lightweight Files for GitHub

In [ ]:
# Transfermarkt (Tier 1), FIFA ratings (Tier 3, optional), 538 SPI (Tier 3, optional --
# note FiveThirtyEight's live site and API are permanently discontinued as of 2023/2025,
# so the Kaggle archive below is the only viable source, not a fallback of convenience),
# Wikipedia biographies (Tier 2, via REST API directly -- the `wikipedia` pip package is
# unreliable for ambiguous names like "Pelé" and is not used here), and blank legacy
# CSV templates for manual RSSSF/IFFHS/WorldFootball coding.

MODERN_DIR = DATA_RAW / "modern"
LEGACY_DIR = DATA_RAW / "legacy"
TM_DIR = MODERN_DIR / "transfermarkt"
FIFA_DIR = DATA_RAW / "optional" / "fifa"
FTE_DIR = DATA_RAW / "optional" / "fivethirtyeight"
WIKI_DIR = LEGACY_DIR / "wikipedia"

# ---- 1. Transfermarkt ----
print("=== 1. Transfermarkt ===")
if not (TM_DIR / "tier1_tm_valuations.parquet").exists():
    print("Downloading Transfermarkt database from Kaggle...")
    !kaggle datasets download davidcariboo/player-scores --unzip -p /content/kaggle_tm -q
    try:
        tm_players = pd.read_csv("/content/kaggle_tm/players.csv")
        tm_vals = pd.read_csv("/content/kaggle_tm/player_valuations.csv")
        mask = tm_players["name"].astype(str).str.contains("|".join(TIER1_PANEL), case=False, na=False)
        panel_tm = tm_players[mask]
        if not panel_tm.empty:
            panel_ids = panel_tm["player_id"].unique()
            panel_vals = tm_vals[tm_vals["player_id"].isin(panel_ids)]
            panel_tm.to_parquet(TM_DIR / "tier1_tm_players.parquet", index=False)
            panel_vals.to_parquet(TM_DIR / "tier1_tm_valuations.parquet", index=False)
            print(f"Saved {len(panel_tm)} profiles, {len(panel_vals)} valuation records.")
        else:
            print("No panel players matched in Transfermarkt data -- check name formatting.")
    except Exception as e:
        print(f"Transfermarkt processing failed: {e}")
    shutil.rmtree("/content/kaggle_tm", ignore_errors=True)
else:
    print("Transfermarkt data already exists on Drive.")

# ---- 2. EA Sports FIFA ratings (optional) ----
print("\n=== 2. EA Sports FC / FIFA Ratings ===")
if not (FIFA_DIR / "tier1_fifa_ratings.parquet").exists():
    print("Downloading EA Sports database from Kaggle...")
    !kaggle datasets download stefanoleone992/ea-sports-fc-24-complete-player-dataset --unzip -p /content/kaggle_fifa -q
    try:
        fifa_df = pd.read_csv("/content/kaggle_fifa/male_players.csv", low_memory=False)
        mask = (
            fifa_df["short_name"].astype(str).str.contains("|".join(TIER1_PANEL), case=False, na=False) |
            fifa_df["long_name"].astype(str).str.contains("|".join(TIER1_PANEL), case=False, na=False)
        )
        panel_fifa = fifa_df[mask]
        print(f"Matched {len(panel_fifa)} rows -- verify this looks reasonable, "
              f"EA's 'short_name' field is often abbreviated (e.g. 'L. Messi').")
        if not panel_fifa.empty:
            panel_fifa.to_parquet(FIFA_DIR / "tier1_fifa_ratings.parquet", index=False)
            print(f"Saved {len(panel_fifa)} historical rating records.")
    except Exception as e:
        print(f"FIFA processing failed: {e}")
    shutil.rmtree("/content/kaggle_fifa", ignore_errors=True)
else:
    print("FIFA ratings already exist on Drive.")

# ---- 3. FiveThirtyEight SPI (optional, archive only -- live source is permanently gone) ----
print("\n=== 3. FiveThirtyEight SPI (Kaggle archive) ===")
if not (FTE_DIR / "spi_global_rankings.csv").exists():
    print("Downloading frozen 538 SPI database from Kaggle...")
    !kaggle datasets download thedevastator/club-soccer-predictions-spi-ratings-and-forecast --unzip -p /content/kaggle_spi -q
    try:
        spi_df = pd.read_csv("/content/kaggle_spi/spi_global_rankings.csv")
        spi_df.to_csv(FTE_DIR / "spi_global_rankings.csv", index=False)
        print(f"Saved 538 SPI rankings: {len(spi_df)} clubs.")
    except Exception as e:
        print(f"538 SPI failed: {e}")
    shutil.rmtree("/content/kaggle_spi", ignore_errors=True)
else:
    print("538 SPI already exists on Drive.")

# ---- 4. Wikipedia biographies (REST API, not the `wikipedia` pip package) ----
print("\n=== 4. Wikipedia Biographies ===")
if not (WIKI_DIR / "goat_biographies.parquet").exists():
    ALL_PLAYERS = TIER1_PANEL + LEGACY_PANEL
    wiki_data = []
    headers = {"User-Agent": "GOAT-Net/1.0 (atix.algo@gmail.com)"}

    for player in ALL_PLAYERS:
        try:
            print(f"  Fetching: {player}...")
            formatted = urllib.parse.quote(player.replace(" ", "_"))
            url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{formatted}"
            res = requests.get(url, headers=headers, timeout=10)

            if res.status_code == 200:
                wiki_data.append({"Player": player, "Biography_Summary": res.json().get("extract")})
            else:
                search_url = (
                    "https://en.wikipedia.org/w/api.php?action=query&list=search"
                    f"&srsearch={urllib.parse.quote(player)}&utf8=&format=json"
                )
                search_res = requests.get(search_url, headers=headers).json()
                results = search_res.get("query", {}).get("search")
                if results:
                    best_match = results[0]["title"]
                    fallback_url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{urllib.parse.quote(best_match.replace(' ', '_'))}"
                    fallback_res = requests.get(fallback_url, headers=headers)
                    if fallback_res.status_code == 200:
                        wiki_data.append({"Player": player, "Biography_Summary": fallback_res.json().get("extract")})
                        print(f"    Resolved via search redirect -> {best_match}")
                else:
                    print(f"    No search results found for {player}.")
            time.sleep(1)
        except Exception as e:
            print(f"    Failed {player}: {e}")

    if wiki_data:
        wiki_df = pd.DataFrame(wiki_data)
        wiki_df.to_parquet(WIKI_DIR / "goat_biographies.parquet", index=False)
        print(f"Saved {len(wiki_df)} biographies to Drive.")
else:
    print("Biographies already exist on Drive.")

# ---- 5. Blank legacy CSV templates for manual RSSSF/IFFHS/WorldFootball coding ----
print("\n=== 5. Legacy CSV Templates ===")
templates = {
    "goat_canon/goat_canon_match_logs_template.csv": ["Player", "Date", "Opponent", "Competition", "Goals", "Assists", "Key_Plays", "Video_Available", "Notes"],
    "rsssf/rsssf_career_goals_template.csv": ["Player", "Year", "Team", "Competition", "Official_Goals", "Friendly_Goals", "Notes"],
    "iffhs/iffhs_awards_template.csv": ["Player", "Year", "Award_Name", "Rank", "Points"],
    "worldfootball/worldfootball_match_history_template.csv": ["Player", "Date", "Team", "Opponent", "Result", "Goals", "Minutes_Played"],
}
for path_suffix, columns in templates.items():
    full_path = LEGACY_DIR / path_suffix
    full_path.parent.mkdir(parents=True, exist_ok=True)
    if not full_path.exists():
        with open(full_path, "w", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow(columns)
        print(f"Created template: {path_suffix}")
    else:
        print(f"Template already exists: {path_suffix}")

print("\nAdditional & optional data sources complete.")

CELL 11

In [ ]:
def build_inventory(directory: Path):
    file_list = []
    for p in directory.rglob("*"):
        if p.is_file():
            file_list.append({
                "relative_path": str(p.relative_to(ROOT)),
                "size_mb": round(p.stat().st_size / (1024 * 1024), 3),
                "last_modified": datetime.datetime.fromtimestamp(p.stat().st_mtime).strftime("%Y-%m-%d %H:%M"),
            })
    return pd.DataFrame(file_list)

inventory_df = build_inventory(DATA_RAW)
inventory_path = META / "dataset_inventory.csv"
inventory_df.to_csv(inventory_path, index=False)

print("\n=== GOAT-Net Vault Inventory ===")
display(inventory_df)

Cell 12

In [ ]:
# ============================================================
# GOAT-Net — Phase 1.5: Rosetta Stone & Dataset Manager Setup
# ============================================================
import os
import pandas as pd
from pathlib import Path

# Paths
ROOT = Path("/content/drive/MyDrive/GOAT-Net")
REPO = Path("/content/GOAT-Net")
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
META = ROOT / "metadata"
SRC_DATA = ROOT / "src" / "data"
SRC_DATA.mkdir(parents=True, exist_ok=True)
(SRC_DATA / "__init__.py").touch()                  # required for `import data.dataset_manager`

for folder in [META, SRC_DATA, DATA_PROCESSED]:
    folder.mkdir(parents=True, exist_ok=True)

print("=== 1. Creating metadata/player_mapping.csv ===")

# Rosetta Stone mapping table for all Tier 1 players across every data provider
player_mapping_data = [
    {
        "canonical_id": "messi",
        "common_name": "Lionel Messi",
        "statsbomb_name": "Lionel Andrés Messi Cuccittini",
        "fbref_name": "Lionel Messi",
        "understat_name": "Lionel Messi",
        "transfermarkt_name": "Lionel Messi",
        "fifa_name": "L. Messi"
    },
    {
        "canonical_id": "ronaldo",
        "common_name": "Cristiano Ronaldo",
        "statsbomb_name": "Cristiano Ronaldo dos Santos Aveiro",
        "fbref_name": "Cristiano Ronaldo",
        "understat_name": "Cristiano Ronaldo",
        "transfermarkt_name": "Cristiano Ronaldo",
        "fifa_name": "Cristiano Ronaldo"
    },
    {
        "canonical_id": "mbappe",
        "common_name": "Kylian Mbappé",
        "statsbomb_name": "Kylian Mbappé Lottin",
        "fbref_name": "Kylian Mbappé",
        "understat_name": "Kylian Mbappé",
        "transfermarkt_name": "Kylian Mbappé",
        "fifa_name": "K. Mbappé"
    },
    {
        "canonical_id": "neymar",
        "common_name": "Neymar",
        "statsbomb_name": "Neymar da Silva Santos Junior",
        "fbref_name": "Neymar",
        "understat_name": "Neymar",
        "transfermarkt_name": "Neymar",
        "fifa_name": "Neymar Jr"
    },
    {
        "canonical_id": "de_bruyne",
        "common_name": "Kevin De Bruyne",
        "statsbomb_name": "Kevin De Bruyne",
        "fbref_name": "Kevin De Bruyne",
        "understat_name": "Kevin De Bruyne",
        "transfermarkt_name": "Kevin De Bruyne",
        "fifa_name": "K. De Bruyne"
    },
    {
        "canonical_id": "lewandowski",
        "common_name": "Robert Lewandowski",
        "statsbomb_name": "Robert Lewandowski",
        "fbref_name": "Robert Lewandowski",
        "understat_name": "Robert Lewandowski",
        "transfermarkt_name": "Robert Lewandowski",
        "fifa_name": "R. Lewandowski"
    },
    {
        "canonical_id": "modric",
        "common_name": "Luka Modrić",
        "statsbomb_name": "Luka Modrić",
        "fbref_name": "Luka Modrić",
        "understat_name": "Luka Modrić",
        "transfermarkt_name": "Luka Modrić",
        "fifa_name": "L. Modrić"
    },
    {
        "canonical_id": "haaland",
        "common_name": "Erling Haaland",
        "statsbomb_name": "Erling Haaland",
        "fbref_name": "Erling Haaland",
        "understat_name": "Erling Haaland",
        "transfermarkt_name": "Erling Haaland",
        "fifa_name": "E. Haaland"
    }
]

mapping_df = pd.DataFrame(player_mapping_data)
mapping_path = META / "player_mapping.csv"
mapping_df.to_csv(mapping_path, index=False)
print(f"✅ Rosetta Stone mapping created at: {mapping_path}")

print("\n=== 2. Creating src/data/dataset_manager.py ===")

dataset_manager_code = '''# GOAT-Net Dataset Manager
# Centralized loader and normalizer for all raw datasets.

import pandas as pd
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/GOAT-Net")
DATA_RAW = ROOT / "data" / "raw"
META = ROOT / "metadata"

def load_player_mapping() -> pd.DataFrame:
    """Loads the canonical player mapping table."""
    mapping_path = META / "player_mapping.csv"
    if not mapping_path.exists():
        raise FileNotFoundError(f"Mapping file not found at {mapping_path}")
    return pd.read_csv(mapping_path)

def normalize_player_names(df: pd.DataFrame, source: str, name_col: str) -> pd.DataFrame:
    """
    Attaches canonical_id and common_name to a dataframe using source-specific name matching.

    Parameters:
        df: The source DataFrame to normalize.
        source: Key matching column in player_mapping.csv ('statsbomb', 'fbref', 'understat', 'transfermarkt', 'fifa').
        name_col: The column in df containing the player's name.
    """
    mapping = load_player_mapping()
    source_key = f"{source.lower()}_name"

    if source_key not in mapping.columns:
        raise ValueError(f"Unknown source '{source}'. Must be one of: statsbomb, fbref, understat, transfermarkt, fifa")

    # Create lookup dictionaries
    id_map = dict(zip(mapping[source_key], mapping["canonical_id"]))
    name_map = dict(zip(mapping[source_key], mapping["common_name"]))

    df = df.copy()
    df["canonical_id"] = df[name_col].map(id_map)
    df["common_name"] = df[name_col].map(name_map)

    return df

def load_statsbomb_events() -> pd.DataFrame:
    """Loads and normalizes StatsBomb spatial event data."""
    path = DATA_RAW / "modern" / "statsbomb" / "statsbomb_panel_events.parquet"
    if not path.exists():
        return pd.DataFrame()
    df = pd.read_parquet(path)
    return normalize_player_names(df, source="statsbomb", name_col="player")

def load_fbref_stats() -> pd.DataFrame:
    """Loads and normalizes all FBref season-level box scores into a single DataFrame."""
    fbref_dir = DATA_RAW / "modern" / "fbref"
    files = list(fbref_dir.glob("*.parquet"))
    if not files:
        return pd.DataFrame()

    dfs = [pd.read_parquet(f) for f in files]
    combined = pd.concat(dfs, ignore_index=True)
    return normalize_player_names(combined, source="fbref", name_col="Player")

def load_understat_xg() -> pd.DataFrame:
    """Loads and normalizes Understat xG metrics."""
    path = DATA_RAW / "modern" / "understat" / "tier1_understat_xg.parquet"
    if not path.exists():
        return pd.DataFrame()
    df = pd.read_parquet(path)
    name_col = "player_name" if "player_name" in df.columns else "Player"
    return normalize_player_names(df, source="understat", name_col=name_col)

def load_transfermarkt() -> pd.DataFrame:
    """Loads and normalizes Transfermarkt player valuations."""
    players_path = DATA_RAW / "modern" / "transfermarkt" / "tier1_tm_players.parquet"
    vals_path = DATA_RAW / "modern" / "transfermarkt" / "tier1_tm_valuations.parquet"

    if not players_path.exists() or not vals_path.exists():
        return pd.DataFrame()

    players = pd.read_parquet(players_path)
    vals = pd.read_parquet(vals_path)

    merged = pd.merge(vals, players, on="player_id", how="left")
    return normalize_player_names(merged, source="transfermarkt", name_col="name")
'''

dm_script_path = SRC_DATA / "dataset_manager.py"
with open(dm_script_path, "w", encoding="utf-8") as f:
    f.write(dataset_manager_code)

print(f"✅ Dataset Manager script written to: {dm_script_path}")

print("\n=== 3. Testing Dataset Manager Loaders ===")
import sys
sys.path.append(str(REPO / "src"))

import importlib
import data.dataset_manager as dm
importlib.reload(dm)

# Test loaders
sb_df = dm.load_statsbomb_events()
fb_df = dm.load_fbref_stats()
us_df = dm.load_understat_xg()
tm_df = dm.load_transfermarkt()

print(f"StatsBomb Events Loaded : {len(sb_df):>6} rows | Unique Players Mapped: {sb_df['canonical_id'].nunique()}")
print(f"FBref Box Scores Loaded  : {len(fb_df):>6} rows | Unique Players Mapped: {fb_df['canonical_id'].nunique()}")
print(f"Understat Records Loaded : {len(us_df):>6} rows | Unique Players Mapped: {us_df['canonical_id'].nunique()}")
print(f"Transfermarkt Valuations : {len(tm_df):>6} rows | Unique Players Mapped: {tm_df['canonical_id'].nunique()}")

print("\n🎉 Phase 1.5 Complete! All data providers now resolve to canonical IDs seamlessly.")

In [ ]:
import os
import sys
import pandas as pd
from pathlib import Path

# Setup Paths
ROOT = Path("/content/drive/MyDrive/GOAT-Net")
sys.path.append(str(REPO / "src"))

print("=== Step 1: Inspecting Player Mapping ===")
mapping_path = ROOT / "metadata" / "player_mapping.csv"
if mapping_path.exists():
    mapping_df = pd.read_csv(mapping_path)
    print(f"✅ Loaded Rosetta Stone with {len(mapping_df)} players.")
    display(mapping_df.head(3))
else:
    print("⚠️ Mapping not found.")

print("\n=== Step 2: Testing Dataset Manager Name Resolution ===")
try:
    from data.dataset_manager import load_fbref_stats, load_understat_xg

    # Check FBref for Haaland
    fbref = load_fbref_stats()
    if not fbref.empty:
        haaland = fbref[fbref["common_name"] == "Erling Haaland"]
        print(f"✅ FBref: Found {len(haaland)} season records for Haaland.")
        print(f"   Raw name used in FBref: {haaland['Player'].unique()[0]}")

    # Check Understat for Messi
    understat = load_understat_xg()
    if not understat.empty:
        messi = understat[understat["common_name"] == "Lionel Messi"]
        print(f"✅ Understat: Found {len(messi)} season records for Messi.")
        player_col = "player_name" if "player_name" in messi.columns else "Player"
        print(f"   Raw name used in Understat: {messi[player_col].unique()[0]}")
except Exception as e:
    print(f"⚠️ Dataset Manager test failed: {e}")

print("\n=== Step 3: FBref Inventory Check ===")
FBREF_DIR = ROOT / "data" / "raw" / "modern" / "fbref"
files = list(FBREF_DIR.glob('*.parquet'))
print(f"📁 Found {len(files)} FBref parquet files.")
for f in sorted(files)[:5]: # Just print the first 5 so it doesn't flood your screen
    size_mb = round(f.stat().st_size / (1024 * 1024), 2)
    print(f"  - {f.name} ({size_mb} MB)")
if len(files) > 5:
    print(f"  ... and {len(files) - 5} more.")

print("\n=== Step 4: How to safely add a player (Pandas 2.0+ Standard) ===")
# Example of how to add a player safely WITHOUT using the deleted .append() method
def add_new_player_safely(canonical_id, common_name, aliases_dict):
    global mapping_df
    if canonical_id in mapping_df["canonical_id"].values:
        print(f"ℹ️ {canonical_id} is already in the database. Skipping.")
        return

    new_row = {"canonical_id": canonical_id, "common_name": common_name, **aliases_dict}
    # Pandas 2.0+ requires pd.concat instead of .append
    mapping_df = pd.concat([mapping_df, pd.DataFrame([new_row])], ignore_index=True)
    mapping_df.to_csv(mapping_path, index=False)
    print(f"✅ Added {common_name} to player_mapping.csv")

# We will test this function by trying to add Haaland (who is already there)
add_new_player_safely(
    canonical_id="haaland",
    common_name="Erling Haaland",
    aliases_dict={"statsbomb_name": "Erling Haaland", "fbref_name": "Erling Haaland", "understat_name": "Erling Haaland", "transfermarkt_name": "Erling Haaland", "fifa_name": "E. Haaland"}
)

In [ ]:
# ============================================================
# GOAT-Net — Phase 0: Data Collection (GitHub Sync)
# ============================================================
import os
import subprocess
from pathlib import Path
from google.colab import userdata
from IPython.display import display, Javascript

# Force Colab to automatically save the notebook right before pushing
display(Javascript('IPython.notebook.save_checkpoint();'))

# We are already in /content/drive/MyDrive/GOAT-Net thanks to Cell 1
REPO = Path("/content/drive/MyDrive/GOAT-Net")

def run_git(args, **kwargs):
    return subprocess.run(["git"] + args, cwd=REPO, capture_output=True, text=True, **kwargs)

try:
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
    if not GITHUB_TOKEN:
        raise ValueError("empty secret")

    REPO_SLUG = "AtiX-Algo/GOAT-Net"
    CLEAN_URL = f"https://github.com/{REPO_SLUG}.git"
    AUTH_URL = f"https://{GITHUB_TOKEN}@github.com/{REPO_SLUG}.git"

    # --- SELF-HEALING CHECK ---
    # If Drive lost the .git folder, re-initialize it instantly
    if not (REPO / ".git").exists():
        print("⚠️ .git folder missing from Drive. Re-initializing repository...")
        run_git(["init"])
        run_git(["remote", "add", "origin", CLEAN_URL])
        run_git(["branch", "-M", "main"])

    # Configure Git
    run_git(["config", "--global", "user.email", "atix.algo@gmail.com"])
    run_git(["config", "--global", "user.name", "AtiX-Algo"])

    # Stage code, docs, and lightweight metadata directly from Drive
    print("📦 Staging files for commit...")
    run_git(["add", "notebooks/"])
    run_git(["add", "src/"])
    run_git(["add", "metadata/"])
    run_git(["add", "docs/"])

    # Explicitly add the panel matches CSV (since it is lightweight and necessary)
    run_git(["add", "-f", "data/raw/modern/statsbomb/statsbomb_panel_matches.csv"])

    # Commit
    commit_res = run_git(["commit", "-m", "feat: update modern datasets, inventory, and dataset manager"])
    if "nothing to commit" in commit_res.stdout:
        print("ℹ️ No new changes to commit. Proceeding to push anyway...")
    else:
        print(commit_res.stdout)

    # FIX GIT STATE & SYNC
    run_git(["remote", "set-url", "origin", AUTH_URL], check=True)

    # Abort any broken rebases
    subprocess.run(["rm", "-rf", ".git/rebase-merge"], cwd=REPO, capture_output=True)
    run_git(["rebase", "--abort"])

    # Force the local 'main' branch to attach to HEAD
    run_git(["branch", "-f", "main", "HEAD"])
    run_git(["checkout", "main"])

    print("\n🔄 Syncing with GitHub (Pulling remote changes)...")
    env = os.environ.copy()
    env["GIT_MERGE_AUTOEDIT"] = "no"

    pull_cmd = ["git", "pull", "origin", "main", "--no-rebase", "-s", "recursive", "-X", "ours", "--allow-unrelated-histories"]
    pull_result = subprocess.run(pull_cmd, cwd=REPO, capture_output=True, text=True, env=env)

    print("🚀 Pushing GOAT-Net updates to GitHub...")
    push_result = run_git(["push", "-u", "origin", "main", "--force"])

    # Strip the token back out immediately for security
    run_git(["remote", "set-url", "origin", CLEAN_URL], check=True)

    if push_result.returncode == 0:
        print("\n✅ Pushed successfully to GitHub. Your repository is now perfectly in sync!")
    else:
        print("\n❌ Push failed. Exact error from GitHub:")
        safe_error = push_result.stderr.replace(GITHUB_TOKEN, "***HIDDEN_TOKEN***")
        print(safe_error)

except Exception as e:
    token_str = GITHUB_TOKEN if 'GITHUB_TOKEN' in locals() and GITHUB_TOKEN else "UNKNOWN_TOKEN"
    safe_error = str(e).replace(token_str, "***HIDDEN_TOKEN***")
    print(f"\n⚠️ GitHub Sync Skipped: {safe_error}")

In [ ]:
# ============================================================
# GOAT-Net Safe GitHub Sync
# Syncs ALL notebooks, source code, metadata, and processed data
# ============================================================
import os, subprocess
from pathlib import Path
from google.colab import userdata
from IPython.display import display, Javascript

# Force Colab to save the notebook right before pushing
display(Javascript('IPython.notebook.save_checkpoint();'))

REPO = Path("/content/drive/MyDrive/GOAT-Net")

# Ensure repo exists
if not REPO.exists():
    raise FileNotFoundError(f"❌ Repository not found at {REPO}. Check your mount and path.")

os.chdir(REPO)

def run_git(args, **kwargs):
    """Run a git command and return the result."""
    return subprocess.run(["git"] + args, cwd=REPO, capture_output=True, text=True, **kwargs)

def print_git_error(cmd_name, result, token=None):
    """Print git errors safely without exposing tokens."""
    error_msg = result.stderr or result.stdout
    if token:
        error_msg = error_msg.replace(token, "***HIDDEN***")
    print(f"⚠️ {cmd_name} warning:\n{error_msg}")

try:
    TOKEN = userdata.get("GITHUB_TOKEN")
    if not TOKEN:
        raise ValueError("❌ GITHUB_TOKEN not set in Colab secrets.")

    REPO_SLUG = "AtiX-Algo/GOAT-Net"
    CLEAN_URL = f"https://github.com/{REPO_SLUG}.git"
    AUTH_URL = f"https://{TOKEN}@github.com/{REPO_SLUG}.git"

    # ============================================
    # 1. Repository Initialization (Safe)
    # ============================================
    if not (REPO / ".git").exists():
        print("🔧 Initializing git repository...")
        run_git(["init"])
        run_git(["remote", "add", "origin", CLEAN_URL])
        run_git(["branch", "-M", "main"])

    # Configure git identity
    run_git(["config", "user.name", "AtiX-Algo"])
    run_git(["config", "user.email", "atix.algo@gmail.com"])
    run_git(["remote", "set-url", "origin", AUTH_URL])

    # ============================================
    # 2. Ensure .gitignore exists
    # ============================================
    gitignore_path = REPO / ".gitignore"
    if not gitignore_path.exists():
        print("📝 Creating .gitignore...")
        gitignore_content = """# Python
__pycache__/
*.pyc
*.pyo
*.egg-info/
dist/
build/

# Jupyter/Colab
.ipynb_checkpoints/
*.ipynb_checkpoints/

# Large raw data files (keep small CSVs, ignore massive parquets)
data/raw/modern/statsbomb/statsbomb_full_match_events.parquet

# Environment and secrets
.env
*.log

# OS files
.DS_Store
Thumbs.db
"""
        gitignore_path.write_text(gitignore_content)

    # ============================================
    # 3. Stage ALL changes (let .gitignore decide)
    # ============================================
    print("📦 Staging all changes...")
    add_result = run_git(["add", "-A"])

    # Check what's being staged
    status_result = run_git(["status", "--short"])
    staged_files = status_result.stdout.strip()

    if staged_files:
        print("\n📋 Files to be committed:")
        print(staged_files)
    else:
        print("\n✨ No changes to commit. Everything is up to date!")
        # Clean up remote URL before exiting
        run_git(["remote", "set-url", "origin", CLEAN_URL])
        raise SystemExit

    # ============================================
    # 4. Commit changes
    # ============================================
    print("\n💾 Committing changes...")
    commit_result = run_git(["commit", "-m", "chore: sync notebooks, source, metadata, and processed data"])

    if commit_result.returncode != 0:
        # Check if it's just "nothing to commit" (safety check)
        if "nothing to commit" in commit_result.stdout or "nothing to commit" in commit_result.stderr:
            print("✨ Nothing to commit after all.")
            run_git(["remote", "set-url", "origin", CLEAN_URL])
            raise SystemExit
        else:
            print_git_error("Commit", commit_result, TOKEN)

    # ============================================
    # 5. Pull latest changes (safe merge)
    # ============================================
    print("\n🔽 Pulling latest changes from GitHub...")

    # First, fetch to see what's on remote
    fetch_result = run_git(["fetch", "origin", "main"])

    # Try a normal pull with merge (not rebase)
    pull_result = run_git(["pull", "origin", "main", "--no-rebase"])

    if pull_result.returncode != 0:
        # Check if there's actually a conflict or if remote is just ahead
        print("ℹ️ Pull had issues. Checking if we need to merge...")

        # Get the difference between local and remote
        run_git(["fetch", "origin"])
        diff_result = run_git(["rev-list", "--count", "HEAD..origin/main"])

        if diff_result.stdout.strip() == "0" or not diff_result.stdout.strip():
            print("✅ Local is up to date with remote. Continuing with push.")
        else:
            print("ℹ️ Attempting merge...")
            merge_result = run_git(["merge", "origin/main", "--no-edit"])
            if merge_result.returncode != 0:
                print("⚠️ Manual merge required. Please resolve conflicts in GitHub.")
                print("   Your changes are committed locally and won't be lost.")
                print(f"   Conflicts: {merge_result.stderr.replace(TOKEN, '***HIDDEN***')}")
                run_git(["remote", "set-url", "origin", CLEAN_URL])
                exit(1)

    # ============================================
    # 6. Push to GitHub (NO force push)
    # ============================================
    print("\n📤 Pushing to GitHub...")
    push_result = run_git(["push", "origin", "main"])  # No --force!

    if push_result.returncode == 0:
        print("✅ Push successful! All your work is securely backed up to GitHub.")
        print(f"   Repository: https://github.com/{REPO_SLUG}")
    elif "non-fast-forward" in push_result.stderr:
        print("⚠️ Push rejected: Remote has changes you don't have locally.")
        print("   This is a SAFETY FEATURE protecting your remote work.")
        print("   Your changes are committed locally.")
        print("   To fix: Run this sync script again to pull and merge first.")
    else:
        print_git_error("Push", push_result, TOKEN)
        print("ℹ️ Your changes are committed locally but couldn't be pushed.")

except Exception as e:
    # Safely print error without exposing token
    error_msg = str(e)
    if 'TOKEN' in locals() and TOKEN:
        error_msg = error_msg.replace(TOKEN, "***HIDDEN***")
    print(f"\n⚠️ GitHub Sync encountered an issue: {error_msg}")
    print("ℹ️ Your notebook and local changes are not affected.")

finally:
    # ============================================
    # ALWAYS clean up: Remove token from remote URL
    # ============================================
    try:
        if (REPO / ".git").exists():
            run_git(["remote", "set-url", "origin", CLEAN_URL])
            print("🔒 Remote URL secured (token removed).")
    except:
        pass  # Silent fail for cleanup

## Notes before running

- **Cell 5 safety limit:** `.head(8)` restricts the first run to 8 competition-seasons so you can confirm the pipeline works before committing to a long scan. Once it runs cleanly and you've checked timing, remove `.head(8)` for the full candidate list.
- **Existence checks are per-file, not per-row.** If a `.parquet`/`.csv` file exists but is incomplete or corrupted (e.g. session died mid-write), the check will skip it as "already exists." If a run looks wrong, delete the specific file on Drive and rerun that cell.
- **`TIER1_PANEL` in Cell 3 matches `docs/PREREGISTRATION.md` §3.** If coverage in Cell 6 is too thin for a control player, log it in the dataset inventory and record any panel change in the pre-registration's Amendments Log — not a silent edit.
- **Cell 8 (FBref/Understat) will still take a while** even with existence checks, since the check only helps on reruns, not the first pass — 70 requests minimum the first time through.
- **Git push (Cell 10) is manual by design now** — review `git status` in Colab before pushing, especially since `data/raw/modern/` may contain more than intended if paths ever drift.

## Next steps after running
1. Check `coverage` (Cell 6) and `inventory` (Cell 9) — log real gaps in the dataset inventory sheet.
2. Once this is stable, extract the repeated "check Drive → download if missing → save" pattern into `src/data/dataset_manager.py` so future notebooks (event processing, statistical engine, etc.) reuse it instead of re-implementing it.
3. Start legacy data collection for Pelé, Maradona, Cruyff, Beckenbauer, Di Stéfano, Puskás — no free structured API exists for them, so this is manual coding into `data/raw/legacy/`.
4. Move to `1_statistical_engine.ipynb`, which should only *load* from Drive/processed data, never download.